In [2]:
import pandas as pd

In [ ]:
import sys
sys.path.append('../game_on/')

from pln_model.limpieza import limpieza

In [5]:
import os
path = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'raw_data', 'steam_games.csv')
df1 = pd.read_csv(path)

In [9]:
data_limpia = limpieza(df1)
data_limpia.columns

Index(['url', 'name', 'release_date', 'popular_tags', 'game_details',
       'languages', 'genre', 'game_description', 'original_price',
       'review_percentage', 'embedding'],
      dtype='object')

MODELO SBERT

In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch

# 1. Configuración del Modelo
# Usamos un modelo balanceado entre velocidad y precisión
model_name = 'all-MiniLM-L6-v2'
model = SentenceTransformer(model_name)

# 2. Simulación de Datos (Basado en el dataset de Steam de Kaggle)
# Cuando el equipo de datos te pase el CSV, cambiaremos esto por pd.read_csv()
data = {
    'game_id': [1, 2, 3, 4, 5],
    'title': ['Left 4 Dead 2', 'Stardew Valley', 'Elden Ring', 'Portal 2', 'Cyberpunk 2077'],
    'description': [
        'Zombies apocalypse cooperative shooter.',
        'Country life RPG with farming and animals.',
        'Challenging action RPG in a dark fantasy world.',
        'Physics-based puzzle game with portals.',
        'Open world action RPG set in a futuristic city.'
    ],
    'tags': ['Action, Zombies, Co-op', 'Farming, Simulation, Relaxing', 'Souls-like, Difficult, RPG', 'Puzzle, Sci-fi, Funny', 'Cyberpunk, RPG, Open World']
}

df = pd.DataFrame(data)

# 3. Creación del "Vibe Text" (Adaptación sugerida)
# SBERT funciona mejor si combinamos la descripción con los tags
df['metadata_combined'] = df['title'] + " " + df['description'] + " " + df['tags']

# 4. Generación de Embeddings
print(f"Vectorizando {len(df)} juegos...")
game_embeddings = model.encode(df['metadata_combined'].tolist(), convert_to_tensor=True)

# 5. Función de Recomendación Adaptada
def get_recommendations(user_query, n_top=3):
    # Vectorizar la consulta del usuario
    query_embedding = model.encode(user_query, convert_to_tensor=True)

    # Calcular similitud coseno
    cosine_scores = util.cos_sim(query_embedding, game_embeddings)[0]

    # Obtener los mejores N resultados
    top_results = torch.topk(cosine_scores, k=n_top)

    print(f"--- Recomendaciones para: '{user_query}' ---\n")
    for score, idx in zip(top_results.values, top_results.indices):
        game = df.iloc[idx.item()]
        print(f"🎯 Juego: {game['title']}")
        print(f"📊 Similitud: {score:.4f}")
        print(f"📝 Tags: {game['tags']}")
        print("-" * 40)

# PRUEBA DE FUNCIONAMIENTO
get_recommendations("I want a difficult game with magic and swords")

/Users/gonzaloneme/.pyenv/versions/3.10.6/envs/Game-On-Project/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6071.01it/s]


Vectorizando 5 juegos...
--- Recomendaciones para: 'I want a difficult game with magic and swords' ---

🎯 Juego: Elden Ring
📊 Similitud: 0.5498
📝 Tags: Souls-like, Difficult, RPG
----------------------------------------
🎯 Juego: Stardew Valley
📊 Similitud: 0.3664
📝 Tags: Farming, Simulation, Relaxing
----------------------------------------
🎯 Juego: Cyberpunk 2077
📊 Similitud: 0.3271
📝 Tags: Cyberpunk, RPG, Open World
----------------------------------------


In [ ]:
import pandas as pd
import numpy as np
import torch
import ast
from sentence_transformers import SentenceTransformer, util

# ==========================================
# 0. DETECCIÓN DE DISPOSITIVO (¡LA CLAVE!)
# ==========================================
# Esto detectará si estás en una Mac con chip M (mps), en un PC con NVIDIA (cuda) o solo CPU
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"🖥️ Dispositivo detectado: {device.type.upper()}")

# ==========================================
# 1. CARGA DEL MODELO
# ==========================================
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
EMBEDDING_DIM = 384

# ==========================================
# 2. CARGA SEGURA DE EMBEDDINGS DESDE LA BASE DE DATOS
# ==========================================
print("🚀 Analizando y reparando formato de embeddings...")

def super_parser(val):
    try:
        # 1. Si es un STRING (texto), intentamos parsearlo
        if isinstance(val, str):
            val = val.replace('\n', '').strip()
            try:
                arr = np.array(ast.literal_eval(val))
            except:
                clean_str = re.sub(r'[\[\]]', '', val)
                arr = np.fromstring(clean_str, sep=' ')

        # 2. Si ya es una LISTA o ARRAY (¡esto era lo que rompía el código antes!)
        elif isinstance(val, (list, np.ndarray)):
            arr = np.array(val)

        # 3. Si es cualquier otra cosa (como un nulo/NaN, que es tipo float)
        else:
            return np.zeros(EMBEDDING_DIM)

        # 4. Verificación final: Debe medir exactamente 384
        if arr.shape == (EMBEDDING_DIM,):
            return arr
        else:
            return np.zeros(EMBEDDING_DIM)

    except:
        # Si algo explota catastróficamente, devolvemos ceros por seguridad
        return np.zeros(EMBEDDING_DIM)

# Aplicamos el super parser
embeddings_list = data_limpia['embedding'].apply(super_parser).tolist()

# Convertimos a tensor y enviamos a la GPU/MPS/CPU
game_embeddings = torch.tensor(np.stack(embeddings_list)).float().to(device)

# --- DIAGNÓSTICO ---
ceros_count = (game_embeddings == 0).all(dim=1).sum().item()
total_juegos = len(game_embeddings)

print(f"\n📊 REPORTE DE LA BASE DE DATOS:")
print(f"   - Juegos totales: {total_juegos}")
print(f"   - Embeddings VÁLIDOS listos para recomendar: {total_juegos - ceros_count}")
print(f"   - Embeddings rotos/vacíos: {ceros_count}")
print("✅ Proceso terminado.\n")

# ==========================================
# 3. FUNCIÓN DE RECOMENDACIÓN
# ==========================================
def recomendar(query, n=5):
    # El modelo vectoriza y ya lo pone en el 'device' automáticamente
    query_vector = model.encode(query, convert_to_tensor=True)

    # Ahora ambos (query_vector y game_embeddings) están en el mismo lugar
    cosine_scores = util.cos_sim(query_vector, game_embeddings)[0]
    top_results = torch.topk(cosine_scores, k=n)

    print(f"\n🎯 Top {n} Recomendaciones para: '{query}'")
    print("="*60)

    for score, idx in zip(top_results.values, top_results.indices):
        juego = data_limpia.iloc[idx.item()]
        print(f"🎮 {juego['name']}")
        print(f"🔥 Match: {score:.2%}")
        print(f"💰 Precio: {juego['original_price']}")
        print(f"🔗 URL: {juego['url']}")
        print("-" * 60)

# ==========================================
# 4. PRUEBA FINAL
# ==========================================
recomendar("A survival game with zombies and crafting")

🖥️ Dispositivo detectado: MPS


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6771.79it/s]


🚀 Analizando y reparando formato de embeddings...

📊 REPORTE DE LA BASE DE DATOS:
   - Juegos totales: 37145
   - Embeddings VÁLIDOS listos para recomendar: 0
   - Embeddings rotos/vacíos: 37145
✅ Proceso terminado.


🎯 Top 5 Recomendaciones para: 'A survival game with zombies and crafting'
🎮 DOOM
🔥 Match: 0.00%
💰 Precio: 19.99
🔗 URL: https://store.steampowered.com/app/379720/DOOM/
------------------------------------------------------------
🎮 PLAYERUNKNOWN'S BATTLEGROUNDS
🔥 Match: 0.00%
💰 Precio: 29.99
🔗 URL: https://store.steampowered.com/app/578080/PLAYERUNKNOWNS_BATTLEGROUNDS/
------------------------------------------------------------
🎮 BATTLETECH
🔥 Match: 0.00%
💰 Precio: 39.99
🔗 URL: https://store.steampowered.com/app/637090/BATTLETECH/
------------------------------------------------------------
🎮 DayZ
🔥 Match: 0.00%
💰 Precio: 44.99
🔗 URL: https://store.steampowered.com/app/221100/DayZ/
------------------------------------------------------------
🎮 EVE Online
🔥 Match: 0.00%
💰 P

In [41]:
import os
from sentence_transformers import SentenceTransformer, util
import torch


path = os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'raw_data', 'steam_games.csv')
df1 = pd.read_csv(path)
data_limpia = limpieza(df1)

# 1. Configuración del Modelo
# Usamos un modelo balanceado entre velocidad y precisión
model_name = 'all-MiniLM-L6-v2'
model = SentenceTransformer(model_name)

data_limpia.columns.tolist()

data_limpia= data_limpia.head(10000)
data_limpia = data_limpia.dropna(subset=['embedding'])

data_limpia['embedding'].isna().sum()

# 1. Primero encodear el texto a vectores numéricos
game_embeddings = model.encode(data_limpia['embedding'].tolist(), convert_to_tensor=True)

query = "action game "
query_embedding = model.encode(query, convert_to_tensor=True)

# 2. Cálculo de similitud coseno
cosine_scores = util.cos_sim(query_embedding, game_embeddings)[0]

# 3. Obtener los mejores N resultados
n_top = 5
top_results = torch.topk(cosine_scores, k=n_top)

# 4. Obtener los mejores N resultados
n_top = 5
top_results = torch.topk(cosine_scores, k=n_top)

# 5. Mostrar resultados
print(f"🔎 Resultados para: '{query}'\n")
print("="*50)

for score, idx in zip(top_results.values, top_results.indices):
    game = data_limpia.iloc[idx.item()]
    print(f"🎮 JUEGO: {game['name']}")
    print(f"📊 Match: {score:.2%}")
    print(f"📂 Género: {game['genre']}")
    print(f"tags: {game['popular_tags']}")
    print(f"💰 Precio: {game['original_price']}")
    print(f"⭐ Reviews: {game['review_percentage']}")
    print(f"🔗 Link: {game['url']}")
    print("-" * 50)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4998.30it/s]


🔎 Resultados para: 'action game '

🎮 JUEGO: Counter-Strike
📊 Match: 54.00%
📂 Género: Genero de juego: Action
tags: Tags populares: Action, FPS, Multiplayer, Shooter, Classic, Team-Based, First-Person, Competitive, Tactical, 1990's, e-sports, PvP, Old School, Military, Strategy, Masterpiece, Survival, Score Attack, 1980s, Assassin
💰 Precio: 9.99
⭐ Reviews: Porcentaje de recomendación de jugadores: 93%
🔗 Link: https://store.steampowered.com/app/10/CounterStrike/
--------------------------------------------------
🎮 JUEGO: Door Kickers: Action Squad
📊 Match: 53.10%
📂 Género: Genero de juego: Action, Casual, Indie, Simulation, Strategy
tags: Tags populares: Action, Indie, Tactical, 2D, Pixel Graphics, Co-op, Retro, Local Co-Op, Multiplayer, Strategy, Zombies, Addictive, Casual, Soundtrack, Replay Value, Level Editor, Real Time Tactics, Moddable, Simulation, Singleplayer
💰 Precio: 13.99
⭐ Reviews: Porcentaje de recomendación de jugadores: 85%
🔗 Link: https://store.steampowered.com/app/686200

**Añadir columna con codigo de imagenes a la base de datos**

In [44]:
data_limpia.head(5)

,url,name,release_date,popular_tags,game_details,languages,genre,game_description,original_price,review_percentage,embedding
0,https://store.steampowered.com/app/379720/DOOM/,DOOM,2016,"Tags populares: FPS, Gore, Action, Demons, Sho...","Tags populares: Single-player, Multi-player, C...","English, French, Italian, German, Spanish - Sp...",Genero de juego: Action,"about this game developed by id software, the ...",19.99,Porcentaje de recomendación de jugadores: 89%,"about this game developed by id software, the ..."
1,https://store.steampowered.com/app/578080/PLAY...,PLAYERUNKNOWN'S BATTLEGROUNDS,2017,"Tags populares: Survival, Shooter, Multiplayer...","Tags populares: Multi-player, Online Multi-Pla...","English, Korean, Simplified Chinese, French, G...","Genero de juego: Action, Adventure, Massively ...",about this game playerunknown's battlegrounds ...,29.99,Porcentaje de recomendación de jugadores: 49%,about this game playerunknown's battlegrounds ...
2,https://store.steampowered.com/app/637090/BATT...,BATTLETECH,2018,"Tags populares: Mechs, Strategy, Turn-Based, T...","Tags populares: Single-player, Multi-player, O...","English, French, German, Russian","Genero de juego: Action, Adventure, Strategy",about this game from original battletech/mechw...,39.99,Porcentaje de recomendación de jugadores: 54%,about this game from original battletech/mechw...
3,https://store.steampowered.com/app/221100/DayZ/,DayZ,2018,"Tags populares: Survival, Zombies, Open World,...","Tags populares: Multi-player, Online Multi-Pla...","English, French, Italian, German, Spanish - Sp...","Genero de juego: Action, Adventure, Massively ...",about this game the post-soviet country of che...,44.99,Porcentaje de recomendación de jugadores: 57%,about this game the post-soviet country of che...
4,https://store.steampowered.com/app/8500/EVE_On...,EVE Online,2003,"Tags populares: Space, Massively Multiplayer, ...","Tags populares: Multi-player, Online Multi-Pla...","English, German, Russian, French","Genero de juego: Action, Free to Play, Massive...",about this game,0.00,Porcentaje de recomendación de jugadores: 54%,"about this game\nGenero de juego: Action, Free..."


In [45]:
# Definir la ruta exacta de tu archivo
ruta_archivo = '/Users/gonzaloneme/Downloads/steam_dataset_2025_csv/applications.csv'

# Cargar el dataset en un DataFrame de Pandas
data_steam = pd.read_csv(ruta_archivo)

# Visualizar las primeras 5 filas del dataset
data_steam.head()

/var/folders/cz/5v_1rcn54x3553fwkbn0fn300000gn/T/ipykernel_94273/698139516.py:5: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  data_steam = pd.read_csv(ruta_archivo)


,appid,name,type,is_free,release_date,required_age,short_description,supported_languages,header_image,background,...,mat_pc_os_min,mat_pc_processor_min,mat_pc_memory_min,mat_pc_graphics_min,mat_pc_os_rec,mat_pc_processor_rec,mat_pc_memory_rec,mat_pc_graphics_rec,created_at,updated_at
0,10,Counter-Strike,game,False,2000-11-01,0,Play the world's number 1 online action game. ...,"English<strong>*</strong>, French<strong>*</st...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
1,20,Team Fortress Classic,game,False,1999-04-01,0,One of the most popular online action games of...,"English, French, German, Italian, Spanish - Sp...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
2,30,Day of Defeat,game,False,2003-05-01,0,Enlist in an intense brand of Axis vs. Allied ...,"English, French, German, Italian, Spanish - Spain",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
3,40,Deathmatch Classic,game,False,2001-06-01,0,Enjoy fast-paced multiplayer gaming with Death...,"English, French, German, Italian, Spanish - Sp...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
4,50,Half-Life: Opposing Force,game,False,1999-11-01,0,Return to the Black Mesa Research Facility as ...,"English, French, German, Korean",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00


In [47]:
data_steam.columns

Index(['appid', 'name', 'type', 'is_free', 'release_date', 'required_age',
       'short_description', 'supported_languages', 'header_image',
       'background', 'metacritic_score', 'recommendations_total',
       'mat_supports_windows', 'mat_supports_mac', 'mat_supports_linux',
       'mat_initial_price', 'mat_final_price', 'mat_discount_percent',
       'mat_currency', 'mat_achievement_count', 'mat_pc_os_min',
       'mat_pc_processor_min', 'mat_pc_memory_min', 'mat_pc_graphics_min',
       'mat_pc_os_rec', 'mat_pc_processor_rec', 'mat_pc_memory_rec',
       'mat_pc_graphics_rec', 'created_at', 'updated_at'],
      dtype='object')

In [ ]:
# 1. Definimos la lista con las columnas exactas que necesitas
columnas_deseadas = [
    'header_image',
    'required_age',
    'short_description',
    'appid',
    'metacritic_score'
]

# 2. Filtramos el DataFrame original y lo guardamos en data_limpia_img
data_limpia_img = data_steam[columnas_deseadas]

# 3. Visualizamos las primeras filas para confirmar que todo está correcto
data_limpia_img.head()

,header_image,required_age,short_description,appid,metacritic_score
0,https://shared.akamai.steamstatic.com/store_it...,0,Play the world's number 1 online action game. ...,10,88.0
1,https://shared.akamai.steamstatic.com/store_it...,0,One of the most popular online action games of...,20,NaN
2,https://shared.akamai.steamstatic.com/store_it...,0,Enlist in an intense brand of Axis vs. Allied ...,30,79.0
3,https://shared.akamai.steamstatic.com/store_it...,0,Enjoy fast-paced multiplayer gaming with Death...,40,NaN
4,https://shared.akamai.steamstatic.com/store_it...,0,Return to the Black Mesa Research Facility as ...,50,NaN


In [49]:
data_limpia.columns

Index(['url', 'name', 'release_date', 'popular_tags', 'game_details',
       'languages', 'genre', 'game_description', 'original_price',
       'review_percentage', 'embedding'],
      dtype='object')

In [50]:
# PASO 1: Extraer el ID numérico de la URL en data_limpia
# Esta línea busca el patrón "/app/números" y guarda solo los números en una nueva columna llamada 'appid'
data_limpia['appid'] = data_limpia['url'].str.extract(r'/app/(\d+)')

# PASO 2: Estandarizar los tipos de datos
# Convertimos la columna 'appid' de AMBOS DataFrames a texto (string) para evitar errores de compatibilidad
data_limpia['appid'] = data_limpia['appid'].astype(str)
data_limpia_img['appid'] = data_limpia_img['appid'].astype(str)

# PASO 3: Unir ambos DataFrames
# Usamos pd.merge y le decimos que use 'appid' como la columna en común.
# how='inner' significa que solo conservará los juegos que existan en AMBAS tablas.
data_final = pd.merge(data_limpia, data_limpia_img, on='appid', how='inner')

# Vemos el resultado final con las columnas combinadas
data_final.head()

/var/folders/cz/5v_1rcn54x3553fwkbn0fn300000gn/T/ipykernel_94273/1769627924.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_limpia_img['appid'] = data_limpia_img['appid'].astype(str)


,url,name,release_date,popular_tags,game_details,languages,genre,game_description,original_price,review_percentage,embedding,appid,header_image,required_age,short_description,metacritic_score
0,https://store.steampowered.com/app/379720/DOOM/,DOOM,2016,"Tags populares: FPS, Gore, Action, Demons, Sho...","Tags populares: Single-player, Multi-player, C...","English, French, Italian, German, Spanish - Sp...",Genero de juego: Action,"about this game developed by id software, the ...",19.99,Porcentaje de recomendación de jugadores: 89%,"about this game developed by id software, the ...",379720,https://shared.akamai.steamstatic.com/store_it...,17,Now includes all three premium DLC packs (Unto...,85.0
1,https://store.steampowered.com/app/578080/PLAY...,PLAYERUNKNOWN'S BATTLEGROUNDS,2017,"Tags populares: Survival, Shooter, Multiplayer...","Tags populares: Multi-player, Online Multi-Pla...","English, Korean, Simplified Chinese, French, G...","Genero de juego: Action, Adventure, Massively ...",about this game playerunknown's battlegrounds ...,29.99,Porcentaje de recomendación de jugadores: 49%,about this game playerunknown's battlegrounds ...,578080,https://shared.akamai.steamstatic.com/store_it...,0,"PUBG: BATTLEGROUNDS, the high-stakes winner-ta...",NaN
2,https://store.steampowered.com/app/637090/BATT...,BATTLETECH,2018,"Tags populares: Mechs, Strategy, Turn-Based, T...","Tags populares: Single-player, Multi-player, O...","English, French, German, Russian","Genero de juego: Action, Adventure, Strategy",about this game from original battletech/mechw...,39.99,Porcentaje de recomendación de jugadores: 54%,about this game from original battletech/mechw...,637090,https://shared.akamai.steamstatic.com/store_it...,0,Take command of your own mercenary outfit of '...,78.0
3,https://store.steampowered.com/app/221100/DayZ/,DayZ,2018,"Tags populares: Survival, Zombies, Open World,...","Tags populares: Multi-player, Online Multi-Pla...","English, French, Italian, German, Spanish - Sp...","Genero de juego: Action, Adventure, Massively ...",about this game the post-soviet country of che...,44.99,Porcentaje de recomendación de jugadores: 57%,about this game the post-soviet country of che...,221100,https://shared.akamai.steamstatic.com/store_it...,17,How long can you survive a post-apocalyptic wo...,NaN
4,https://store.steampowered.com/app/8500/EVE_On...,EVE Online,2003,"Tags populares: Space, Massively Multiplayer, ...","Tags populares: Multi-player, Online Multi-Pla...","English, German, Russian, French","Genero de juego: Action, Free to Play, Massive...",about this game,0.00,Porcentaje de recomendación de jugadores: 54%,"about this game\nGenero de juego: Action, Free...",8500,https://shared.akamai.steamstatic.com/store_it...,0,EVE Online is a community driven space MMO whe...,88.0


In [52]:
data_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2254 entries, 0 to 2253
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   url                2254 non-null   object 
 1   name               2254 non-null   object 
 2   release_date       2254 non-null   int64  
 3   popular_tags       2254 non-null   object 
 4   game_details       2254 non-null   object 
 5   languages          2254 non-null   object 
 6   genre              2254 non-null   object 
 7   game_description   2254 non-null   object 
 8   original_price     2254 non-null   float64
 9   review_percentage  2254 non-null   object 
 10  embedding          2254 non-null   object 
 11  appid              2254 non-null   object 
 12  header_image       2254 non-null   object 
 13  required_age       2254 non-null   object 
 14  short_description  2254 non-null   object 
 15  metacritic_score   1017 non-null   float64
dtypes: float64(2), int64(1),